In [1]:
SRC = "/kaggle/input/trendyol-src/src"
DATA = "/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle"
WORK = "/kaggle/working"

In [2]:
  !pip install -q rapidfuzz text-unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.9 MB/s eta 0:00:0000:0100:01


In [3]:
  from pathlib import Path
  import shutil, os, sys

  DATA_DIR = Path("/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle")
  SRC_IN = Path("/kaggle/input/datasets/z3lkahmet/baseline-scripts")
  SRC = Path("/kaggle/working/src")
  WORK = Path("/kaggle/working")

  if not DATA_DIR.exists():
      DATA_DIR = Path("/kaggle/input/trendyol-e-ticaret-yarismasi-2026-kaggle")

  shutil.copytree(SRC_IN, SRC, dirs_exist_ok=True)
  sys.path.insert(0, str(SRC))

  print(DATA_DIR)
  print(SRC)

/kaggle/input/competitions/trendyol-e-ticaret-yarismasi-2026-kaggle
/kaggle/working/src


In [4]:
 !python /kaggle/working/src/train_term_negatives.py \
    --data-dir {DATA_DIR} \
    --output /kaggle/working/train_term_negatives.csv

terms=17,968 items=962,873 positives=250,000 vocab=250,000
mine train-term negatives: 100%|██████████████| 141/141 [00:28<00:00,  4.93it/s]
wrote /kaggle/working/train_term_negatives.csv
negatives=872,163 hard=435,462 mid=257,021 easy=179,680


In [5]:
!python /kaggle/working/src/sample_negative_audit.py \
    --data-dir {DATA_DIR} \
    --negatives /kaggle/working/train_term_negatives.csv \
    --output /kaggle/working/negative_audit_sample.csv \
    --n-per-band 200

loaded_negatives=872,163
positive_overlap=0
wrote /kaggle/working/negative_audit_sample.csv rows=600
negative_band
easy    200
hard    200
mid     200


In [6]:
  !python /kaggle/working/src/transformer_biencoder.py train \
    --data-dir {DATA_DIR} \
    --negatives /kaggle/working/train_term_negatives.csv \
    --model-dir /kaggle/working/transformer_biencoder \
    --epochs 2 \
    --batch-size 64 \
    --fp16

dropped_negative_positive_overlap=0
training_rows=1,122,163 positives=250,000 positive_rate=0.2228
load item texts: 9it [00:30,  3.36s/it]
tokenizer_config.json: 100%|███████████████████| 526/526 [00:00<00:00, 2.07MB/s]
tokenizer.json: 100%|██████████████████████| 9.08M/9.08M [00:00<00:00, 26.9MB/s]
special_tokens_map.json: 100%|█████████████████| 239/239 [00:00<00:00, 1.62MB/s]
model.safetensors: 100%|██████████████████████| 471M/471M [00:02<00:00, 180MB/s]
Loading weights: 100%|█| 199/199 [00:00<00:00, 1590.27it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/kaggle/working/src/transformer_biencoder.py:294: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. P

In [ ]:
  !python /kaggle/working/src/transformer_biencoder.py predict \
    --data-dir {DATA_DIR} \
    --model-dir /kaggle/input/datasets/z3lkahmet/bi-encoder \
    --output /kaggle/working/transformer_biencoder_scores.csv \
    --device cuda

device=cuda:0 device_ids=[0, 1]
Loading weights: 100%|█| 199/199 [00:00<00:00, 338.74it/s, Materializing param=p
using DataParallel for encoder on cuda devices [0, 1]
using DataParallel for head on cuda devices [0, 1]
unique_terms=32,185 unique_items=929,781
load item texts: 9it [00:48,  5.34s/it]
encode texts:  44%|███████████▍              | 799/1816 [13:48<17:54,  1.06s/it]

In [ ]:
  !find /kaggle/working -maxdepth 3 -type f | sort | grep -E "score|catboost|lgbm|transformer|csv"
  !find /kaggle/input -maxdepth 4 -type f | sort | grep -E "score|catboost|lgbm|transformer|csv"

In [7]:
!python /kaggle/working/src/ensemble_scores.py blend \
    --catboost /kaggle/input/datasets/z3lkahmet/outputs/catboost_category_aware_2026-06-30_scores.csv \
    --lgbm /kaggle/input/datasets/z3lkahmet/outputs/pu_lgbm_v1_scores.csv \
    --transformer /kaggle/working/transformer_biencoder_scores.csv \
    --output /kaggle/working/moe_gbdt_rankblend_scores.csv \
    --report-correlations

wrote /kaggle/working/moe_gbdt_rankblend_scores.csv rows=3,359,679
weights gbdt=(0.700 catboost, 0.300 lgbm) final=(0.500 gbdt, 0.500 transformer)
rank correlation matrix
                  catboost_rank  lgbm_rank  transformer_rank  gbdt_rank   prob
catboost_rank            1.0000     0.8670            0.7578     0.9883 0.9262
lgbm_rank                0.8670     1.0000            0.7514     0.9314 0.8926
transformer_rank         0.7578     0.7514            1.0000     0.7772 0.9427
gbdt_rank                0.9883     0.9314            0.7772     1.0000 0.9427
prob                     0.9262     0.8926            0.9427     0.9427 1.0000


In [14]:
  !python /kaggle/working/src/ensemble_scores.py submit \
    --scores /kaggle/working/moe_gbdt_rankblend_scores.csv \
    --output-dir /kaggle/working \
    --rates 0.18 0.20 0.22 0.24

/kaggle/working/submission_moe_gbdt_rankblend_r18.csv: rows=3,359,679 positives=604,742 rate=0.18000
/kaggle/working/submission_moe_gbdt_rankblend_r20.csv: rows=3,359,679 positives=671,936 rate=0.20000
/kaggle/working/submission_moe_gbdt_rankblend_r22.csv: rows=3,359,679 positives=739,129 rate=0.22000
/kaggle/working/submission_moe_gbdt_rankblend_r24.csv: rows=3,359,679 positives=806,323 rate=0.24000


In [8]:
  !python /kaggle/working/src/ensemble_scores.py submit \
    --scores /kaggle/working/moe_gbdt_rankblend_scores.csv \
    --output-dir /kaggle/working \
    --rates 0.24 0.25 0.26 0.27 0.28 0.30

/kaggle/working/submission_moe_gbdt_rankblend_r24.csv: rows=3,359,679 positives=806,323 rate=0.24000
/kaggle/working/submission_moe_gbdt_rankblend_r25.csv: rows=3,359,679 positives=839,920 rate=0.25000
/kaggle/working/submission_moe_gbdt_rankblend_r26.csv: rows=3,359,679 positives=873,517 rate=0.26000
/kaggle/working/submission_moe_gbdt_rankblend_r27.csv: rows=3,359,679 positives=907,113 rate=0.27000
/kaggle/working/submission_moe_gbdt_rankblend_r28.csv: rows=3,359,679 positives=940,710 rate=0.28000
/kaggle/working/submission_moe_gbdt_rankblend_r30.csv: rows=3,359,679 positives=1,007,904 rate=0.30000


In [5]:
  !python src/ensemble_scores.py submit \
    --scores /kaggle/input/datasets/z3lkahmet/outputs/moe_gbdt_rankblend_scores.csv \
    --output-dir /kaggle/working \
    --prefix submission_moe_constrained \
    --rates 0.25 0.26 0.27 \
    --min-positive-excess-base 100

/kaggle/working/submission_moe_constrained_r25.csv: rows=3,359,679 positives=839,920 rate=0.25000 min_positive_excess_base=100
/kaggle/working/submission_moe_constrained_r26.csv: rows=3,359,679 positives=873,517 rate=0.26000 min_positive_excess_base=100
/kaggle/working/submission_moe_constrained_r27.csv: rows=3,359,679 positives=907,113 rate=0.27000 min_positive_excess_base=100


In [9]:
  for w in [0.55, 0.60, 0.65, 0.70]:
      out = f"/kaggle/working/moe_blend_t{int(w*100)}.csv"
      !python /kaggle/working/src/ensemble_scores.py blend \
        --catboost /kaggle/input/datasets/z3lkahmet/outputs/catboost_category_aware_2026-06-30_scores.csv \
        --lgbm /kaggle/input/datasets/z3lkahmet/outputs/pu_lgbm_v1_scores.csv \
        --transformer /kaggle/working/transformer_biencoder_scores.csv \
        --output {out} \
        --gbdt-weight {1-w} \
        --transformer-weight {w}

wrote /kaggle/working/moe_blend_t55.csv rows=3,359,679
weights gbdt=(0.700 catboost, 0.300 lgbm) final=(0.450 gbdt, 0.550 transformer)
wrote /kaggle/working/moe_blend_t60.csv rows=3,359,679
weights gbdt=(0.700 catboost, 0.300 lgbm) final=(0.400 gbdt, 0.600 transformer)
wrote /kaggle/working/moe_blend_t65.csv rows=3,359,679
weights gbdt=(0.700 catboost, 0.300 lgbm) final=(0.350 gbdt, 0.650 transformer)
wrote /kaggle/working/moe_blend_t70.csv rows=3,359,679
weights gbdt=(0.700 catboost, 0.300 lgbm) final=(0.300 gbdt, 0.700 transformer)


In [4]:
  from pathlib import Path

  score_files = {
      "t55": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t55.csv",
      "t60": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t60.csv",
      "t65": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t65.csv",
      "t70": "/kaggle/input/datasets/z3lkahmet/moe-blends/moe_blend_t70.csv",
  }

  for name, path in score_files.items():
      assert Path(path).exists(), path
      !python /kaggle/working/src/ensemble_scores.py submit \
        --scores {path} \
        --output-dir /kaggle/working \
        --prefix submission_moe_blend_{name} \
        --rates 0.24 0.25 0.26 0.27 0.28

/kaggle/working/submission_moe_blend_t55_r24.csv: rows=3,359,679 positives=806,323 rate=0.24000
/kaggle/working/submission_moe_blend_t55_r25.csv: rows=3,359,679 positives=839,920 rate=0.25000
/kaggle/working/submission_moe_blend_t55_r26.csv: rows=3,359,679 positives=873,517 rate=0.26000
/kaggle/working/submission_moe_blend_t55_r27.csv: rows=3,359,679 positives=907,113 rate=0.27000
/kaggle/working/submission_moe_blend_t55_r28.csv: rows=3,359,679 positives=940,710 rate=0.28000
/kaggle/working/submission_moe_blend_t60_r24.csv: rows=3,359,679 positives=806,323 rate=0.24000
/kaggle/working/submission_moe_blend_t60_r25.csv: rows=3,359,679 positives=839,920 rate=0.25000
/kaggle/working/submission_moe_blend_t60_r26.csv: rows=3,359,679 positives=873,517 rate=0.26000
/kaggle/working/submission_moe_blend_t60_r27.csv: rows=3,359,679 positives=907,113 rate=0.27000
/kaggle/working/submission_moe_blend_t60_r28.csv: rows=3,359,679 positives=940,710 rate=0.28000
/kaggle/working/submission_moe_blend_t65

In [15]:
 !tar -czf /kaggle/working/transformer_biencoder.tar.gz -C /kaggle/working transformer_biencoder

In [10]:
!ls /kaggle/working

moe_blend_t55.csv		       submission_moe_gbdt_rankblend_r25.csv
moe_blend_t60.csv		       submission_moe_gbdt_rankblend_r26.csv
moe_blend_t65.csv		       submission_moe_gbdt_rankblend_r27.csv
moe_blend_t70.csv		       submission_moe_gbdt_rankblend_r28.csv
moe_gbdt_rankblend_scores.csv	       submission_moe_gbdt_rankblend_r30.csv
negative_audit_sample.csv	       train_term_negatives.csv
src				       transformer_biencoder_scores.csv
submission_moe_gbdt_rankblend_r24.csv


In [14]:
  !cd /kaggle/working && rm -f working_no_src.zip && zip -r working_no_src.zip . -x "src/*" "./src/*" "working_no_src.zip" && ls -lh
  working_no_src.zip

  adding: moe_blend_t60.csv (deflated 53%)
  adding: submission_moe_gbdt_rankblend_r25.csv (deflated 55%)
  adding: moe_blend_t70.csv (deflated 53%)
  adding: transformer_biencoder_scores.csv (deflated 54%)
  adding: .virtual_documents/ (stored 0%)
  adding: moe_blend_t65.csv (deflated 53%)
  adding: submission_moe_gbdt_rankblend_r24.csv (deflated 55%)
  adding: moe_gbdt_rankblend_scores.csv (deflated 53%)
  adding: negative_audit_sample.csv (deflated 72%)
  adding: submission_moe_gbdt_rankblend_r30.csv (deflated 55%)
  adding: train_term_negatives.csv (deflated 71%)
  adding: submission_moe_gbdt_rankblend_r28.csv (deflated 55%)
  adding: moe_blend_t55.csv (deflated 53%)
  adding: submission_moe_gbdt_rankblend_r26.csv (deflated 55%)
  adding: submission_moe_gbdt_rankblend_r27.csv (deflated 55%)
total 1.9G
-rw-r--r-- 1 root root 140M Jul  9 11:19 moe_blend_t55.csv
-rw-r--r-- 1 root root 140M Jul  9 11:20 moe_blend_t60.csv
-rw-r--r-- 1 root root 140M Jul  9 11:20 moe_blend_t65.csv
-rw-r-

NameError: name 'working_no_src' is not defined

In [5]:
!kaggle competitions submit -c trendyol-e-ticaret-yarismasi-2026-kaggle -f /kaggle/working/submission_moe_blend_t65_r28.csv  -m "transformers blend"

100%|██████████████████████████████████████| 67.3M/67.3M [00:01<00:00, 50.9MB/s]
Successfully submitted to Trendyol E-Ticaret Yarışması 2026 - Kaggle